# Thí nghiệm bổ sung sau review: hoán đổi α và seed 2 cho ablation

Repo: https://github.com/KienNguyenDev2711/Hyena-Attention-Study

Notebook này **không chứa logic thí nghiệm**. Mọi logic nằm trong `hyena_study/followup.py`,
có test đường dây `tests/test_followup.py`. Notebook chỉ clone repo, chạy test rồi gọi module.

**Chạy hai tài khoản song song:** tài khoản A đặt `LANG = "vi"`, tài khoản B đặt `LANG = "en"`.

| LANG | Nội dung nếu corpus khớp báo cáo | Ước lượng T4 |
|---|---|---|
| `vi` | 3 run hoán đổi `E4x_vi_alphaen` + 5 run ablation seed 2 | ~10 phút token hoá + ~80 phút |
| `en` | 3 run hoán đổi `E4x_en_alphavi` | ~12 phút token hoá + ~30 phút |

Nếu **dấu vân tay corpus lệch** (do phiên bản `datasets` khác), module tự chạy thêm 3 run đối chứng
`E4c` cùng phiên và **bỏ qua** ablation seed 2. Không cần sửa gì, chỉ cần đọc dòng in ra ở bước 4.

Trước khi chạy: Settings → Accelerator **GPU T4**, Internet **On**. Nên dùng **Save Version → Save & Run All**
để phiên chạy nền, đóng trình duyệt không bị ngắt. Chạy lại notebook sẽ bỏ qua các run đã xong.


In [ ]:
LANG  = "vi"   # tài khoản thứ hai đổi thành "en"
EXTRA = ""     # "--with_control" để chạy thêm đối chứng E4c cùng phiên (khuyến nghị cho "en")

## 1 · Nạp mã nguồn

In [ ]:
REPO_URL = "https://github.com/KienNguyenDev2711/Hyena-Attention-Study.git"
WORK     = "/kaggle/working/Hyena-Attention-Study"

import os, subprocess, sys

os.chdir("/kaggle/working")
if not os.path.isdir(WORK):
    subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, WORK], check=True)
else:
    subprocess.run(["git", "-C", WORK, "pull", "-q"], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 2 · Thư viện và GPU

`datasets==4.0.0` là phiên bản đã tái lập được corpus tiếng Anh của báo cáo (ghi trong `results/E4_corpus_en_s0.json`).

In [ ]:
!pip install -q "datasets==4.0.0" tokenizers

import torch, datasets
print("torch", torch.__version__, "· datasets", datasets.__version__)
assert torch.cuda.is_available(), "CHUA BAT GPU: Settings -> Accelerator -> GPU T4"
print("GPU:", torch.cuda.get_device_name(0))

## 3 · Test đường dây (CPU, khoảng 1–2 phút). Có dòng FAIL thì DỪNG, không chạy tiếp.

In [ ]:
r = subprocess.run([sys.executable, "tests/test_followup.py"], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "test_followup THAT BAI - dung lai, gui log cho nhom"

## 4 · Kế hoạch và dấu vân tay corpus

Đọc kỹ dòng `DAU VAN TAY KHOP` hoặc `CORPUS KHAC VOI BAO CAO`.

In [ ]:
!python -m hyena_study.followup --lang {LANG} {EXTRA} --dry_run

In [ ]:
!python -m hyena_study.followup --lang {LANG} --prepare_only

## 5 · Huấn luyện (dài). Chết giữa chừng thì chạy lại ô này, run đã xong sẽ được bỏ qua.

In [ ]:
!python -m hyena_study.followup --lang {LANG} {EXTRA}

## 6 · Đóng gói kết quả

Tải file zip ở panel Output rồi gửi lại cho nhóm.

In [ ]:
import json, shutil
man = json.load(open(f"results_followup/followup_{LANG}_manifest.json", encoding="utf-8"))
print("corpus khop bao cao:", man["corpus_matches"], "| mismatches:", man["mismatches"])
for r in man["runs"]:
    print(f"  {r['kind']:<9}{r['name']:<26}PPL {r['test_ppl']:.3f}  ({r['status']})")
out = shutil.make_archive(f"/kaggle/working/followup_{LANG}_results", "zip", "results_followup")
print("Da dong goi:", out)